# 05 · Building similarity and learned models (Phase 6)

Given a building at A, where are locations B with similar surroundings? And how well do surroundings predict where building happens?

1. reference set (P6.2);
2. **single-building queries A → B** with a per-feature explanation (P6.1, P6.3);
3. **aggregate similarity** of every usable cell (P6.4);
4. random forests: change RF and state RF, spatial-block CV (P6.5);
5. held-out building check (P6.6);
6. comparison of the three scores.

Build: `python -m src.pipeline --from-stage similarity --to-stage similarity` (or the modules one by one). Checked by `scripts/verify_similarity.py` and `scripts/verify_similarity_models.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.config import load_config

pd.set_option("display.max_colwidth", 90)
cfg = load_config()
MET = Path(cfg.paths["metrics"])
FIG = Path(cfg.paths["figures"])


def show(name, width=760):
    display(Image(filename=str(FIG / f"{name}.png"), width=width))

## 1. Reference set

Cells with an OSM building, plus ESRI built cells where OSM has none (OSM maps 62–75 % of built cells in the north but ~1 % in the south in 2018).

In [ ]:
from src.similarity.reference import buildings, reference_path

rows = []
for run in ("validation", "final"):
    ref = pd.read_parquet(reference_path(cfg, run))
    rows.append({"run": run, "reference cells": len(ref), **ref["source"].value_counts().to_dict(),
                 "OSM buildings": int(ref["n_buildings"].sum())})
pd.DataFrame(rows).set_index("run")

In [ ]:
b = buildings(cfg, "final")
b["type_group"].value_counts(normalize=True).round(4).to_frame("share of current OSM buildings")

Almost every OSM building is untyped (`building=yes`), so similarity *per building type* (P6.7) isn't possible on this data. The queries below still pick typed buildings, so the examples differ.

## 2. Single-building queries

Profile = the cell's surroundings (`schema.MODEL_INPUTS`: land-cover shares in the 250 m and 500 m rings, distances to built-up, water and roads, road density, terrain), z-scored on reference + candidate cells. Distance = Euclidean (decision D5). Results are usable cells of the final 3-class map only.

In [ ]:
from src.similarity import query as q
from src.similarity.profiles import run_profiles

rp = run_profiles(cfg, "final")
examples = []
for t in ("industrial", "apartments", "school", "house"):
    pick = b.loc[b["building_type"] == t].sample(1, random_state=cfg["project"]["random_seed"])
    examples.append((t, int(pick["bldg_id"].iloc[0])))
examples

In [ ]:
results = {}
for t, bid in examples:
    res = q.query(cfg, "final", bldg_id=bid, n=20, profiles=rp)
    q.write_query(cfg, res, f"{t}_{bid}")
    results[t] = res
    print(f"== {res.label}")
    display(res.results[["rank", "cell_id", "distance", "similarity", "closest_features", "most_different"]].head(5))

In [ ]:
for t, bid in examples:
    show(f"map_similarity_query_{t}_{bid}", 620)

What matches between A and its best match B (features sorted by the gap in standard deviations; shares that are ~0 at both cells come last):

In [ ]:
q.explain(results["industrial"], cfg).round(2)

**Euclidean vs cosine.** Cosine ignores magnitude, so a cell whose profile points the same way but is much more extreme counts as identical. Overlap of the two top-20 lists:

In [ ]:
for t, bid in examples:
    cos = q.query(cfg, "final", bldg_id=bid, n=20, metric="cosine", profiles=rp)
    common = len(set(cos.results["cell_id"]) & set(results[t].results["cell_id"]))
    print(f"{t:11s}: {common} of 20 cells in common")

## 3. Aggregate similarity

Mean distance of every usable cell to its 10 nearest reference profiles (a reference cell is never matched with itself), rescaled to 0–1.

In [ ]:
sim = json.loads((MET / "similarity.json").read_text())
print("AUC on grew (validation run):", round(sim["similarity_knn"]["temporal_preview"]["auc_grew"], 3))
show("map_score_similarity_knn_final", 600)

## 4. Random forests

In [ ]:
pd.DataFrame(
    {
        m: {
            "training label": sim[m]["training"]["label"],
            "training cells": sim[m]["training"]["n_cells"],
            "positives": sim[m]["training"]["n_positive"],
            "best params": sim[m]["best_params"],
            "spatial-CV AUC (training label)": round(sim[m]["spatial_cv_auc"]["mean"], 3),
            "AUC on grew, out of fold": round(sim[m]["temporal_preview"]["auc_grew_oof"], 3),
        }
        for m in ("rf_change", "rf_state")
    }
)

The state RF separates built from non-built almost perfectly (0.998), mostly through distance to built-up, but that skill doesn't carry over to *where growth happens next*. The change RF learns from cells that actually changed. Scored on its own training cells it would claim 0.997, which is why validation scores are out of fold (D11).

In [ ]:
show("chart_rf_importance")

## 5. Held-out building check

In [ ]:
h = sim["holdout"]["validation"]["scorers"]
pd.DataFrame(h).T[["auc", "median_percentile", "best_possible_median", "share_top10", "n_held_out"]].round(3)

Medians are capped near 70 because reference cells are ~60 % of the non-excluded cells, so AUC is the fair measure. Keeping both ring scales helps (0.895 vs 0.871 with 500 m rings only).

## 6. Comparison

In [ ]:
v = json.loads((MET / "validation.json").read_text())["models"]
pd.DataFrame({m: {"AUC": v[m]["auc"], "lift top 10 %": v[m]["lift_top10"]} for m in
              ("similarity_knn", "rf_change", "rf_state", "dist_to_built")}).T.round(3)

In [ ]:
show("chart_roc", 520)

**Reading it.** The aggregate similarity and the change RF reach almost the same AUC (0.773 vs 0.779) by different routes: one never sees a growth label, the other learns from 2020/21 change. Both beat distance to built-up, the bar to beat, and both rank cells next to existing built-up land first. Phase 7 combines them.